***Scans browser and selector from hdf5 file***


Author: J.-S. Micha

Last Revision:   June  2026

**Objectives**

- create a `logfile` object to explore an hdf5file

- build as many scan dictionaries corresponding to each identified scan in hdh5 indexed scans list (`dfallscans` pandas dataframe object)
e.g. d87 = logfile.build_dict_scan(87) for scan index in `dfallscans`. Each dictionary contains needed info to proceed with further suited anaysis workflow

- `logfile` contains two useful methods:

filterby() and findstrings() to locate interesting scans

- with the function argument openGUI=True or selectscansGUI(): scans can be selected by clicking on button. The list scan dictionary will be recorded in `logfile.selectedscans`

- getscansfromdate() allows to find scans by date

# SET environment

In [ ]:
JUPYTER_LAB = True # False for jupyter hub or notebook

# at ESRF during experiment and 90 days after at /data/visitor
# OR for a long term project at /data/projects
DATA_AT_ESRF_NICE=True   # False

# IMPORT packages

In [ ]:
if not JUPYTER_LAB:
    %matplotlib notebook  # jupyterhub
else:
    %matplotlib widget

import os, time, copy, sys
import glob

import matplotlib.pyplot as plt
import matplotlib

import numpy as np

import ipywidgets
from IPython.display import display
from ipywidgets import interact, interactive, fixed, interact_manual

from pathlib import Path
import pandas as pd

In [ ]:
if 0:  # only for venv environment, not conda !!
    import sys
    devfolder = '/data/bm32/inhouse/lauetoolsenv2/lib/python3.12/site-packages'
    sys.path.insert(0,devfolder)
    import LaueTools as LT
    LT.__file__

In [ ]:
import LaueTools.generaltools as GT
import LaueTools.blissdatafolderstructure as blissfolders
import LaueTools.logfile_reader as iohdf5
import LaueTools
print('you are using LaueTools of the following folder (of possibly an environment)', LaueTools.__path__)

In [ ]:
if 0: # test if we can get some plots (depending on notebook version ... and ipympl needs or not)
    fig_,ax_ = plt.subplots()
    ax_.plot(np.arange(20))

# SET user-defined `ExperimentFolder`

parent folder of all raw data ot the experiment. For data at ESRF during and after experiment, the path should contain 'RAW_DATA'

In [ ]:
#Experiment number or id?   the subfolder of date will be added if there is one
#expId = 'ma6030', 'hc5386', 'a322860', 'a322864'

# expId = 'ihma714'
# CCDLabel = 'sCMOS'

expId = 'ma6758'
CCDLabel = 'EIGER_4MCdTe'

expId = 'blc17179'
CCDLabel = 'EIGER_4MCdTe'

ExperimentFolder = blissfolders.getExperimentFolder(expId, data_at_esrf=DATA_AT_ESRF_NICE)
ExperimentFolder

# Browsing data folders and building rapidly scan dictionnary `d`.

`d` to be used in next section

## list of folders and files

In [ ]:
blissfolders.tree(ExperimentFolder,0)

In [ ]:
blissfolders.tree(Path(ExperimentFolder)/'BaTiO3',0, truncatesize=15)

In [ ]:
# use the master (uppest level) h5 file
pathHDF5, HDF5_LOGFILE_EXISTS = blissfolders.findmasterh5file(ExperimentFolder)
print(pathHDF5, HDF5_LOGFILE_EXISTS)

# or use a lower level h5 file
subfolder ='BaTiO3'
pathHDF5_subfile,  HDF5_LOGFILE_EXISTS= blissfolders.findmasterh5file(os.path.join(ExperimentFolder,subfolder))
print(pathHDF5_subfile,  HDF5_LOGFILE_EXISTS)

## read the hdf5 file: 

In [ ]:
#logfile = iohdf5.H5file(pathHDF5, CCDLabel)
logfile = iohdf5.H5file(pathHDF5_subfile, CCDLabel)
print('selected logfile .h5 :',logfile.path)

In [ ]:
logfile.getscans(node_substrings=['map2D'],verbose=1)

In [ ]:
logfile.dfallscans

## Select scan or list of scans

fill dictionary `logfile.selectedscans`

### [OPTION] use GUI scan selector

In [ ]:
logfile.selectscansGUI(nmax=10)

In [ ]:
print('current selected scan index in dataframe scans list',logfile.selectedscansindices)

In [ ]:
logfile.selectedscans.keys()

In [ ]:
if logfile.selectedscansindices:
    print('first selected scan in the selectedscans list\n\n',logfile.selectedscans[logfile.selectedscansindices[0]])


## Manual build of dictionary of scan parameters

In [ ]:
_=logfile.build_dict_scan(87);
logfile.build_dict_scan(4);
logfile.build_dict_scan(2000);

In [ ]:
logfile.selectedscans

## HDF5 mining to look for specific scans

### Location of mesh scans (2D map)

In [ ]:
selectedscans = logfile.findstrings('250nm','', openGUI=False)  # default column1 = sample_dataset_scanindex, column2 = fullcommand
selectedscans

In [ ]:
print('current selected scan index in dataframe scans list',logfile.selectedscansindices)


### Location of zf or thf scans (DAXM)

In [ ]:
logfile.filterby('motors', 'zf', openGUI=False)

In [ ]:
print('current selected scan index in dataframe scans list',logfile.selectedscansindices)


### Location of scans for specified sample

In [ ]:
dfsample = logfile.findstrings('250nm','', openGUI=False)
dfsample

In [ ]:
print('current selected scan index in dataframe scans list',logfile.selectedscansindices)


In [ ]:
logfile.dfallscans

### retrieve bliss command and data from image and `dfallscans` from hdf5 file

In [ ]:
logfile.selectedscans

In [ ]:
ccdlabel_from_dict = next(iter(logfile.selectedscans.values())).get('CCDLabel', None)
print('ccdlabel_from_dict',ccdlabel_from_dict)

if ccdlabel_from_dict == 'sCMOS':
    exampleimage = os.path.join(d87['folder'],'img_0000.tif')
    #exampleimage = os.path.join(logfile.selectedscans[87]['folder'],'eiger4m_0000.h5')
    print('image', exampleimage)
    
    blisscommand, imagedate, logfile_scanindex, df_found = logfile.getscanfromimage(exampleimage, CCDLabel='sCMOS')
    
elif ccdlabel_from_dict == 'EIGER_4MCdTe':
    d= logfile.build_dict_scan(4)
    exampleimage = os.path.join(d['folder'],'eiger4m_0000.h5')
    blisscommand, imagedate, logfile_scanindex , df_found= logfile.getscanfromimage(exampleimage, CCDLabel='EIGER_4MCdTe',
                                                                          timespan_minutes=200)
df_found

In [ ]:
df_found.iloc[0], df_found.iloc[0].endreason


#### Example 1: Select the scan collecting images at querydate

In [ ]:
dfallscansgooddate = logfile.getscansfromdate((2026,6,20,17,31,0),timespan=200, verbose=True)
dfallscansgooddate

#### Example 2: Select the scan collecting images at during the period

In [ ]:
dfallscansgoodperiod = logfile.getscansfromdate((2026,3,17,4,33,30), timespan=60)
dfallscansgoodperiod

### Rapid access to image folder

In [ ]:
# get images_subfolder from dataframe index and fullpath image folder
logfile_index = 5
df=logfile.dfallscans  # choose your dataframe from hdf5 filtered or not ... dfzfscans, dfallscans

#*********************************
#to see a specific item property  use ** loc **
fullpath_imagefolder = df.loc[logfile_index]['imagefolder']

images_subfolder = blissfolders.setimages_subfolder(fullpath_imagefolder, rootfolder='RAW_DATA')
print('for item #%d'%logfile_index)
print('images_subfolder is (to be copied afterwards if needed):')
print(images_subfolder)

# FINAL SELECTION

- either you use the scans properties dictionary from above: `logfile.selectedscans`

- or you build you own lists from a list of indices and rebuild `logfile.selectedscans`

In [ ]:
logfile.selectedscansindices, logfile.selectedscans

In [ ]:
# ******MANUAL SELECTIONS **************
index_userlist = [1000,3,4,5,6,88,102]

for idx in index_userlist:
    if idx >= len(logfile.dfallscans):
        GT.printyellow(f'this index {idx} is not an entry (row) in logfile.dfallscans pandas dataframe')
        continue
    logfile.build_dict_scan(idx)

In [ ]:
logfile.selectedscans